In [3]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

train_data = pd.read_csv("hmm_pos_train_dataset.csv")
test_data = pd.read_csv("hmm_pos_test_dataset.csv")

print("Training Dataset")
print(train_data.head())

print("\nTest Dataset")
print(test_data.head())

train_words = train_data["word"].astype(str).tolist()
train_tags = train_data["pos"].astype(str).tolist()

test_words = test_data["word"].astype(str).tolist()
test_tags = test_data["pos"].astype(str).tolist()

print("\nTraining samples:", len(train_words))
print("Test samples:", len(test_words))

transition_counts = defaultdict(Counter)

previous_tag = "<START>"

for tag in train_tags:
    transition_counts[previous_tag][tag] += 1
    previous_tag = tag

transition_counts[previous_tag]["<END>"] += 1

transition_probabilities = defaultdict(dict)

for previous_tag in transition_counts:
    total = sum(transition_counts[previous_tag].values())

    for tag in transition_counts[previous_tag]:
        transition_probabilities[previous_tag][tag] = (
            transition_counts[previous_tag][tag] / total
        )

emission_counts = defaultdict(Counter)
tag_counts = Counter()

for word, tag in zip(train_words, train_tags):
    word = word.lower()
    emission_counts[tag][word] += 1
    tag_counts[tag] += 1

emission_probabilities = defaultdict(dict)

for tag in emission_counts:
    total = sum(emission_counts[tag].values())

    for word in emission_counts[tag]:
        emission_probabilities[tag][word] = (
            emission_counts[tag][word] / total
        )

tags = sorted(tag_counts.keys())
vocabulary = set(word.lower() for word in train_words)

print("\nPOS Tags:", tags)
print("Vocabulary size:", len(vocabulary))

def get_transition_probability(previous_tag, current_tag):
    if current_tag in transition_probabilities[previous_tag]:
        return transition_probabilities[previous_tag][current_tag]

    return 1 / (
        sum(transition_counts[previous_tag].values())
        + len(tags)
        + 1
    )

def get_emission_probability(tag, word):
    word = word.lower()

    if word in emission_probabilities[tag]:
        return emission_probabilities[tag][word]

    return 1 / (
        tag_counts[tag]
        + len(vocabulary)
        + 1
    )

def viterbi(sentence):
    words = sentence.split()

    if len(words) == 0:
        return []

    viterbi_table = [{}]
    backpointer = [{}]

    first_word = words[0]

    for tag in tags:
        transition = get_transition_probability("<START>", tag)
        emission = get_emission_probability(tag, first_word)

        viterbi_table[0][tag] = (
            np.log(transition) + np.log(emission)
        )

        backpointer[0][tag] = None

    for i in range(1, len(words)):
        word = words[i]

        viterbi_table.append({})
        backpointer.append({})

        for current_tag in tags:
            best_score = None
            best_previous_tag = None

            emission = get_emission_probability(
                current_tag,
                word
            )

            for previous_tag in tags:
                transition = get_transition_probability(
                    previous_tag,
                    current_tag
                )

                score = (
                    viterbi_table[i - 1][previous_tag]
                    + np.log(transition)
                    + np.log(emission)
                )

                if best_score is None or score > best_score:
                    best_score = score
                    best_previous_tag = previous_tag

            viterbi_table[i][current_tag] = best_score
            backpointer[i][current_tag] = best_previous_tag

    best_tag = max(
        viterbi_table[-1],
        key=viterbi_table[-1].get
    )

    predicted_tags = [best_tag]

    for i in range(len(words) - 1, 0, -1):
        best_tag = backpointer[i][best_tag]
        predicted_tags.append(best_tag)

    predicted_tags.reverse()

    return list(zip(words, predicted_tags))

sentence = "The student reads a book"

result = viterbi(sentence)

print("\nPOS Tagging Result")
print("------------------")

for word, tag in result:
    print(word, "→", tag)

test_sentence = " ".join(test_words)

result = viterbi(test_sentence)

predicted_tags = [tag for word, tag in result]

correct = 0
total = len(test_tags)

for actual, predicted in zip(test_tags, predicted_tags):
    if actual == predicted:
        correct += 1

accuracy = correct / total

print("\nTest Dataset Evaluation")
print("-----------------------")
print("Correct predictions:", correct)
print("Total predictions:", total)
print("Test Dataset Accuracy:", round(accuracy * 100, 2), "%")

actual_counter = Counter(test_tags)

report = []

for tag in tags:
    true_positive = 0
    false_positive = 0
    false_negative = 0

    for actual, predicted in zip(test_tags, predicted_tags):

        if actual == tag and predicted == tag:
            true_positive += 1

        elif actual != tag and predicted == tag:
            false_positive += 1

        elif actual == tag and predicted != tag:
            false_negative += 1

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else 0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else 0
    )

    f1_score = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    report.append([
        tag,
        precision,
        recall,
        f1_score,
        actual_counter[tag]
    ])

evaluation_report = pd.DataFrame(
    report,
    columns=[
        "POS Tag",
        "Precision",
        "Recall",
        "F1-Score",
        "Support"
    ]
)

print("\nEvaluation Report")
print("-----------------")
print(evaluation_report.to_string(index=False))

macro_precision = evaluation_report["Precision"].mean()
macro_recall = evaluation_report["Recall"].mean()
macro_f1 = evaluation_report["F1-Score"].mean()

print("\nOverall Evaluation")
print("------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(macro_precision, 4))
print("Recall   :", round(macro_recall, 4))
print("F1-Score :", round(macro_f1, 4))

sentence = input("\nEnter a sentence: ")

result = viterbi(sentence)

print("\nPredicted POS Tags")
print("------------------")

for word, tag in result:
    print(word, "→", tag)

Training Dataset
      word   pos
0      The   DET
1  student  NOUN
2    reads  VERB
3        a   DET
4     book  NOUN

Test Dataset
      word   pos
0      The   DET
1  student  NOUN
2    reads  VERB
3        a   DET
4     book  NOUN

Training samples: 50
Test samples: 24

POS Tags: ['ADJ', 'ADP', 'ADV', 'AUX', 'DET', 'NOUN', 'PRON', 'PROPN', 'PUNCT', 'VERB']
Vocabulary size: 38

POS Tagging Result
------------------
The → DET
student → NOUN
reads → VERB
a → DET
book → NOUN

Test Dataset Evaluation
-----------------------
Correct predictions: 24
Total predictions: 24
Test Dataset Accuracy: 100.0 %

Evaluation Report
-----------------
POS Tag  Precision  Recall  F1-Score  Support
    ADJ        1.0     1.0       1.0        1
    ADP        0.0     0.0       0.0        0
    ADV        0.0     0.0       0.0        0
    AUX        1.0     1.0       1.0        1
    DET        1.0     1.0       1.0        4
   NOUN        1.0     1.0       1.0        6
   PRON        1.0     1.0       1.


Enter a sentence:  The student reads a book



Predicted POS Tags
------------------
The → DET
student → NOUN
reads → VERB
a → DET
book → NOUN
